# YOLO26s Training Pipeline

## 1. Setup & Configuration

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

# Device detection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"Using device:    {DEVICE}")

In [ ]:
# Paths — absolute to avoid CWD issues
BASE_DIR = Path(r"D:/school/school/Final Project/ENG402 Graduation Project/model_building")
DATASET_DIR = BASE_DIR / "datasets" / "dataset1" / "data"
DATASET_YAML = DATASET_DIR / "dataset.yaml"

# ======================================================================
# Training hyperparameters
# ======================================================================

# --- Model & project ---
MODEL_NAME = "yolo26s.pt"
PROJECT = str(BASE_DIR / "model_train" / "runs")
EXPERIMENT = "yolo26s_shapes"

# --- Training control ---
EPOCHS = 50
BATCH_SIZE = 32
IMG_SIZE = 640
PATIENCE = 20  # early stopping patience
SEED = 42
DETERMINISTIC = True
AMP = True
WORKERS = 0
FRACTION = 1.0

# --- Learning rate schedule ---
LR0 = 0.01
LRF = 0.1
WARMUP_EPOCHS = 3
WARMUP_BIAS_LR = 0.1
WARMUP_MOMENTUM = 0.8

# --- Loss weights ---
BOX = 7.5
CLS = 0.5
DFL = 1.5  # distribution focal loss (YOLOv8+ replaces obj)

# --- Regularization ---
WEIGHT_DECAY = 0.01
DROPOUT = 0.2

# --- Segmentation masks ---
OVERLAP_MASK = True
MASK_RATIO = 4

# --- Augmentation ---
FLIPLR = 0.5
FLIPUD = 0.0
DEGREES = 10.0
SCALE = 0.5
TRANSLATE = 0.1
SHEAR = 2.0
PERSPECTIVE = 0.0005
HSV_H = 0.01
HSV_S = 0.7
HSV_V = 0.4
MOSAIC = 1.0
MIXUP = 0.1
COPY_PASTE = 0.2
AUTO_AUGMENT = 'autoaugment'

# ======================================================================
# Configuration summary
# ======================================================================
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Dataset:       {DATASET_YAML}")
print(f"Model:         {MODEL_NAME}")
print(f"Experiment:    {EXPERIMENT}")
print(f"Device:        {DEVICE}")

print("\n--- Training control ---")
print(f"Epochs:        {EPOCHS}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Image size:    {IMG_SIZE}")
print(f"Patience:      {PATIENCE}")
print(f"Seed:          {SEED}  (deterministic={DETERMINISTIC})")
print(f"AMP:           {AMP}")
print(f"Workers:       {WORKERS}")
print(f"Fraction:      {FRACTION}")

print("\n--- LR schedule ---")
print(f"lr0 / lrf:     {LR0} / {LRF}")
print(f"Warmup:        {WARMUP_EPOCHS} epochs (bias_lr={WARMUP_BIAS_LR}, momentum={WARMUP_MOMENTUM})")

print("\n--- Loss weights ---")
print(f"box/cls/dfl:   {BOX} / {CLS} / {DFL}")

print("\n--- Regularization ---")
print(f"weight_decay:  {WEIGHT_DECAY}")
print(f"dropout:       {DROPOUT}")

print("\n--- Augmentation ---")
print(f"flip (lr/ud):  {FLIPLR} / {FLIPUD}")
print(f"degrees:       {DEGREES}")
print(f"scale:         {SCALE}")
print(f"translate:     {TRANSLATE}")
print(f"shear:         {SHEAR}")
print(f"perspective:   {PERSPECTIVE}")
print(f"hsv (h/s/v):   {HSV_H} / {HSV_S} / {HSV_V}")
print(f"mosaic:        {MOSAIC}")
print(f"mixup:         {MIXUP}")
print(f"copy_paste:    {COPY_PASTE}")
print(f"auto_augment:  {AUTO_AUGMENT}")
print("=" * 60)

## 2. Dataset Verification

In [ ]:
import yaml

# Verify dataset.yaml exists and print contents
assert DATASET_YAML.exists(), f"dataset.yaml not found at {DATASET_YAML}"

with open(DATASET_YAML) as f:
    ds_config = yaml.safe_load(f)

print("dataset.yaml contents:")
for k, v in ds_config.items():
    print(f"  {k}: {v}")

# Count samples per split
for split in ["train", "val", "test"]:
    img_dir = DATASET_DIR / ds_config[split]
    lbl_dir = DATASET_DIR / ds_config[split].replace("images", "labels")
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"\n{split}: {n_img} images, {n_lbl} labels")
    if n_img != n_lbl:
        print(f"  WARNING: mismatch ({n_img} images vs {n_lbl} labels)")

## 3. Training

In [ ]:
# Load pretrained YOLO26s
model = YOLO(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
# Train
results = model.train(
    # --- Data & output ---
    data=str(DATASET_YAML),
    project=PROJECT,
    name=EXPERIMENT,
    exist_ok=True,

    # --- Training control ---
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    seed=SEED,
    deterministic=DETERMINISTIC,
    amp=AMP,
    workers=WORKERS,
    fraction=FRACTION,
    pretrained=True,
    verbose=True,

    # --- LR schedule ---
    lr0=LR0,
    lrf=LRF,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_bias_lr=WARMUP_BIAS_LR,
    warmup_momentum=WARMUP_MOMENTUM,

    # --- Loss weights ---
    box=BOX,
    cls=CLS,
    dfl=DFL,

    # --- Regularization ---
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,

    # --- Segmentation masks ---
    overlap_mask=OVERLAP_MASK,
    mask_ratio=MASK_RATIO,

    # --- Augmentation ---
    fliplr=FLIPLR,
    flipud=FLIPUD,
    degrees=DEGREES,
    scale=SCALE,
    translate=TRANSLATE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    auto_augment=AUTO_AUGMENT,
)

## 4. Validation

In [ ]:
# Load best weights from training
best_weights = Path(PROJECT) / EXPERIMENT / "weights" / "best.pt"
assert best_weights.exists(), f"Best weights not found at {best_weights}"

best_model = YOLO(str(best_weights))
print(f"Loaded best weights: {best_weights}")

In [ ]:
# Validate on val split
val_results = best_model.val(
    data=str(DATASET_YAML),
    split="val",
    device=DEVICE,
    verbose=True,
)

print(f"\n--- Validation Results ---")
print(f"mAP50:    {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

## 5. Testing & Metrics

In [ ]:
# Test on test split
test_results = best_model.val(
    data=str(DATASET_YAML),
    split="test",
    device=DEVICE,
    verbose=True,
)

print(f"\n--- Test Results ---")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP50-95:  {test_results.box.map:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

In [ ]:
# Per-class metrics
class_names = ds_config["names"]
print("\nPer-class AP50:")
for i, ap in enumerate(test_results.box.ap50):
    print(f"  {class_names[i]:>12s}: {ap:.4f}")

In [ ]:
# Display training curves (saved by ultralytics during training)
from IPython.display import Image as IPImage, display

results_dir = Path(PROJECT) / EXPERIMENT

for plot_name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f"\n{plot_name}:")
        display(IPImage(filename=str(plot_path)))

## 6. Export

In [ ]:
# Summary of all saved artifacts
print("Training artifacts:")
results_dir = Path(PROJECT) / EXPERIMENT
for p in sorted(results_dir.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  {p.relative_to(results_dir)}  ({size_mb:.1f} MB)")